In [0]:
CATALOG, SCHEMA, VOLUME = "workspace", "olist", "raw_files"
path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
display(dbutils.fs.ls(path))

path,name,size,modificationTime
dbfs:/Volumes/workspace/olist/raw_files/olist_customers_dataset.csv,olist_customers_dataset.csv,9033957,1795115892000
dbfs:/Volumes/workspace/olist/raw_files/olist_order_items_dataset.csv,olist_order_items_dataset.csv,15438671,1795115994000
dbfs:/Volumes/workspace/olist/raw_files/olist_order_payments_dataset.csv,olist_order_payments_dataset.csv,5777138,1795115880000
dbfs:/Volumes/workspace/olist/raw_files/olist_orders_dataset.csv,olist_orders_dataset.csv,17654914,1795116034000
dbfs:/Volumes/workspace/olist/raw_files/olist_products_dataset.csv,olist_products_dataset.csv,2379446,1795115862000
dbfs:/Volumes/workspace/olist/raw_files/product_category_name_translation.csv,product_category_name_translation.csv,2613,1795115697000


In [0]:
from pyspark.sql import functions as F

files = {
    "bronze_orders":               "olist_orders_dataset.csv",
    "bronze_order_items":          "olist_order_items_dataset.csv",
    "bronze_products":             "olist_products_dataset.csv",
    "bronze_customers":            "olist_customers_dataset.csv",
    "bronze_payments":             "olist_order_payments_dataset.csv",
    "bronze_category_translation": "product_category_name_translation.csv",
}

for table, fname in files.items():
    df = (spark.read
          .option("header", True)
          .csv(f"{path}/{fname}")
          .withColumn("_ingested_at", F.current_timestamp()))
    (df.write.format("delta").mode("overwrite")
       .saveAsTable(f"{CATALOG}.{SCHEMA}.{table}"))
    print("loaded", table)

loaded bronze_orders
loaded bronze_order_items
loaded bronze_products
loaded bronze_customers
loaded bronze_payments
loaded bronze_category_translation


In [0]:
for t in files:
    print(t, spark.table(f"{CATALOG}.{SCHEMA}.{t}").count())

spark.table(f"{CATALOG}.{SCHEMA}.bronze_orders").printSchema()

bronze_orders 99441
bronze_order_items 112650
bronze_products 32951
bronze_customers 99441
bronze_payments 103886
bronze_category_translation 71
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)

